# Weather Data Visualisation
**Course:** Data Science Undergraduate Pipeline Project  
**Data source:** Open-Meteo API — Colombo, Sri Lanka  

This notebook connects to PostgreSQL, loads all stored weather records,
and produces time-series charts for the report.


In [ ]:
# Imports
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print('Libraries loaded.')

In [ ]:
# Database connection parameters
# When running on the host machine, use localhost:5432.
# The PostgreSQL container exposes port 5432 to the host.

DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'dbname':   'weather_db',
    'user':     'airflow',
    'password': 'airflow',
}

conn = psycopg2.connect(**DB_CONFIG)
print('Connected to PostgreSQL successfully.')

In [ ]:
# Load data into a DataFrame 

query = """
    SELECT
        id,
        extraction_timestamp,
        city,
        temperature_2m,
        relative_humidity_2m,
        wind_speed_10m,
        surface_pressure,
        weather_code
    FROM weather_data
    ORDER BY extraction_timestamp ASC;
"""

df = pd.read_sql(query, conn)

# Ensure the timestamp column is a proper datetime type
df['extraction_timestamp'] = pd.to_datetime(df['extraction_timestamp'])

print(f'Loaded {len(df)} records.')
df.head()

In [ ]:
# Basic statistics 

df.describe()

In [ ]:
# Chart 1 — Temperature over time 

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(
    df['extraction_timestamp'],
    df['temperature_2m'],
    marker='o',
    color='tomato',
    linewidth=1.5,
    markersize=4,
    label='Temperature (°C)'
)

ax.set_title('Temperature at 2 m — Colombo, Sri Lanka', fontsize=14, fontweight='bold')
ax.set_xlabel('Date / Time (UTC)')
ax.set_ylabel('Temperature (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate()
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../screenshots/chart_temperature.png', dpi=150)
plt.show()
print('Chart saved to screenshots/chart_temperature.png')

In [ ]:
# Chart 2 — Wind speed over time 

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(
    df['extraction_timestamp'],
    df['wind_speed_10m'],
    marker='s',
    color='steelblue',
    linewidth=1.5,
    markersize=4,
    label='Wind Speed (km/h)'
)

ax.set_title('Wind Speed at 10 m — Colombo, Sri Lanka', fontsize=14, fontweight='bold')
ax.set_xlabel('Date / Time (UTC)')
ax.set_ylabel('Wind Speed (km/h)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate()
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../screenshots/chart_wind_speed.png', dpi=150)
plt.show()
print('Chart saved to screenshots/chart_wind_speed.png')

In [ ]:
# Chart 3 — Relative humidity over time 

fig, ax = plt.subplots(figsize=(12, 4))

ax.fill_between(
    df['extraction_timestamp'],
    df['relative_humidity_2m'],
    alpha=0.4,
    color='teal',
    label='Humidity (%)'
)
ax.plot(
    df['extraction_timestamp'],
    df['relative_humidity_2m'],
    color='teal',
    linewidth=1.5
)

ax.set_title('Relative Humidity at 2 m — Colombo, Sri Lanka', fontsize=14, fontweight='bold')
ax.set_xlabel('Date / Time (UTC)')
ax.set_ylabel('Relative Humidity (%)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate()
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../screenshots/chart_humidity.png', dpi=150)
plt.show()
print('Chart saved to screenshots/chart_humidity.png')

In [ ]:
# Close the connection

conn.close()
print('Database connection closed.')